<a href="https://colab.research.google.com/github/priyal6/agent-evaluation/blob/main/contextual_relevancy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q deepeval openai numpy

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.33.0 requires click<9.0.0,>=8.4.2, but you have click 8.3.3 which is incompatible.


In [2]:
from google.colab import userdata

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

In [3]:
documents = [
    "Our return policy allows customers to return items within 30 days of purchase for a full refund.",
    "Refunds are processed to the original payment method within 5-7 business days.",
    "Shipping is free on all orders over $50 within the continental US.",
    "International shipping takes 10-15 business days and costs a flat $25.",
    "Our customer support team is available Monday to Friday, 9am to 6pm EST.",
    "Gift cards are non-refundable and cannot be exchanged for cash.",
]

In [6]:
import numpy as np
from openai import OpenAI

client = OpenAI(api_key = OPENAI_API_KEY)
EMBED_MODEL = "text-embedding-3-small"
MODEL = "gpt-5.6"

def embed(texts):
  resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
  vecs = np.array([d.embedding for d in resp.data])
  return vecs/ np.linalg.norm(vecs, axis=1, keepdims=True)

doc_vectors = embed(documents)

In [12]:
def retrieve(query, k=3):
    q = embed([query])[0]
    scores = doc_vectors @ q
    top_idx = np.argsort(scores)[::-1][:k]
    return [documents[i] for i in top_idx]

def generate(query, context_chunks):
    context = "\n".join(f"- {c}" for c in context_chunks)
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Answer only using the provided context. If the answer isn't there, say so."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
        ],
    )
    return resp.choices[0].message.content

def rag(query, k=3):
    chunks = retrieve(query, k)
    return generate(query, chunks), chunks


answer, chunks = rag("How long do refunds take?")
print(answer)
chunks

Refunds are processed to the original payment method within 5–7 business days.


['Refunds are processed to the original payment method within 5-7 business days.',
 'Our return policy allows customers to return items within 30 days of purchase for a full refund.',
 'Gift cards are non-refundable and cannot be exchanged for cash.']

contextual relevancy

In [15]:
import os
from deepeval.metrics import ContextualRelevancyMetric
from deepeval.test_case import LLMTestCase


os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

metric = ContextualRelevancyMetric(
    threshold=0.7,
    model="gpt-4o-mini",
    include_reason=True,
    async_mode=False,
)

query = "How long do refunds take?"
answer, chunks = rag(query)


test_case = LLMTestCase(
    input=query,
    actual_output=answer,
    retrieval_context=chunks,
)

metric.measure(test_case)
print("Score:", metric.score)
print("Reason:", metric.reason)

Output()

Score: 0.6666666666666666
Reason: The score is 0.67 because while the context includes relevant information such as 'Refunds are processed to the original payment method within 5-7 business days,' it also contains irrelevant details about gift cards that do not address the question about refund timelines.


In [16]:
from deepeval import evaluate

questions = [
    "How long do refunds take?",
    "Is shipping free?",
    "Can I get cash back for a gift card?",
    "When can I contact support?",
]

test_cases = []
for q in questions:
    ans, ctx = rag(q)
    test_cases.append(LLMTestCase(input=q, actual_output=ans, retrieval_context=ctx))

results = evaluate(test_cases=test_cases, metrics=[metric])

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:            How long do refunds take?                                                              │
│  │     Actual Output:    Refunds are processed to the original payment method within 5–7 business days.         │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Contextual Relevancy │ 0.67  │ 0.70      │ The score is 0.67 because while the context           │
│              │                      │       │           │ mentions that 'Refunds are processed to the           │
│              │                      │       │           │ original payment method within 5-7 business days,'    │
│              │                      │       │           │ it also includes irrelevant information about gift    │
│              │                      │       │           │ cards being non-refundable, which detracts from the   │
│              │                      │       │           │ overall relevance.                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:            Is shipping free?                                                                      │
│  │     Actual Output:    Shipping is free on orders over $50 within the continental US. International           │
│  │                       shipping costs a flat $25.                                                             │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Contextual Relevancy │ 0.33  │ 0.70      │ The score is 0.33 because while the relevant          │
│              │                      │       │           │ statement notes that 'Shipping is free on all         │
│              │                      │       │           │ orders over $50 within the continental US', the       │
│              │                      │       │           │ context also includes irrelevant information like     │
│              │                      │       │           │ 'costs a flat $25', which contradicts the idea of     │
│              │                      │       │           │ free shipping.                                        │
│                                                          


⚠ WARNING: No hyperparameters logged.
» Log hyperparameters to attribute prompts and models to your test runs.



✓ Evaluation completed 🎉! (time taken: 52.24s | token cost: 0.0096618 USD)
» Test Results (4 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 4

= 

» Want to share evals with your team, or a place for your test cases to live? ❤️
🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.




In [17]:
import pandas as pd

rows = []
for tc in test_cases:
    m = ContextualRelevancyMetric(threshold=0.7, model="gpt-4o-mini", async_mode=False)
    m.measure(tc)
    rows.append({"question": tc.input, "score": round(m.score, 2), "passed": m.is_successful(), "reason": m.reason})

pd.DataFrame(rows)

Output()

Output()

Output()

Output()

,question,score,passed,reason
0,How long do refunds take?,0.67,False,The score is 0.67 because while the context in...
1,Is shipping free?,0.33,False,The score is 0.33 because while the relevant s...
2,Can I get cash back for a gift card?,0.33,False,The score is 0.33 because while the relevant s...
3,When can I contact support?,0.33,False,The score is 0.33 because while the relevant s...
